# 3P Judge Inference for Error Localization

This notebook runs error localization prompts through a 3P model (via MetaGen API).

## Workflow
1. Export prompts: `python export_error_localization_prompts.py --experiments "..." --output prompts.jsonl`
2. Run this notebook to get judge responses
3. Match results: `python match_error_localization_results.py --prompts prompts.jsonl --responses responses.jsonl`

## Cell 1: Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - MODIFY THESE
# =============================================================================

# Input/Output paths
INPUT_JSONL = "prompts.jsonl"      # From export_error_localization_prompts.py
OUTPUT_JSONL = "responses.jsonl"   # For match_error_localization_results.py
CHECKPOINT_JSONL = "checkpoint.jsonl"  # Resume support

# Model settings
MODEL_NAME = "llama3.1-405b-instruct"  # Model to use as judge
metagen_key = None  # SET YOUR API KEY HERE

# Inference settings
BATCH_SIZE = 10       # Number of prompts per batch (concurrent API calls)
MAX_TOKENS = 1024     # Max tokens for response
TEMPERATURE = 0.3     # Low temp for consistent judging
TOP_P = 0.9
TOP_K = 50

# Retry settings (for API errors only)
MAX_RETRIES = 3

print(f"Input: {INPUT_JSONL}")
print(f"Output: {OUTPUT_JSONL}")
print(f"Model: {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE}")

## Cell 2: MetaGen Setup

In [ ]:
import json
import asyncio
import logging
from pathlib import Path
from tqdm import tqdm

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# MetaGen imports
import metagen.bento
from metagen import CompletionResponse, Message, MetaGenKey, MetaGenPlatform

# Create MetaGen platform
metagen_platform: MetaGenPlatform = metagen.bento.create_metagen_platform(
    MetaGenKey(key=metagen_key)
)

print("✓ MetaGen platform initialized")

In [ ]:
from libfb.py.asyncio.await_utils import async_run

async def batch_inference_async(prompts, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    """Async batch inference with retry on API errors."""
    
    async def infer_single_with_retry(prompt, retry_count=0):
        """Single inference with exponential backoff retry for API errors."""
        try:
            messages = Message.message_list().add_user_message(prompt).build()
            response: CompletionResponse = await metagen_platform.chat_completion_async(
                model=MODEL_NAME,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=TOP_P,
                top_k=TOP_K,
                messages=messages,
            )
            return response.choices[0].text
        except Exception as e:
            if retry_count < MAX_RETRIES:
                wait_time = 2 ** retry_count  # Exponential backoff: 1s, 2s, 4s
                logger.warning(f"API error (attempt {retry_count + 1}/{MAX_RETRIES + 1}): {e}. Retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
                return await infer_single_with_retry(prompt, retry_count + 1)
            else:
                logger.error(f"API failed after {MAX_RETRIES + 1} attempts: {e}")
                raise
    
    tasks = [infer_single_with_retry(prompt) for prompt in prompts]
    completions = await asyncio.gather(*tasks, return_exceptions=True)
    return completions


def batch_inference(prompts, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    """Synchronous wrapper for batch inference."""
    return async_run(batch_inference_async(prompts, temperature, max_tokens))


print("✓ Inference functions ready")

## Cell 3: Load Prompts

In [ ]:
def load_jsonl(filepath):
    """Load JSONL file into list of dicts."""
    entries = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries


def load_completed_ids(checkpoint_path):
    """Load set of already-completed row_ids from checkpoint."""
    if not Path(checkpoint_path).exists():
        return set()
    
    completed = set()
    with open(checkpoint_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                entry = json.loads(line)
                completed.add(entry['row_id'])
    return completed


# Load prompts
print(f"Loading prompts from: {INPUT_JSONL}")
all_prompts = load_jsonl(INPUT_JSONL)
print(f"Total prompts: {len(all_prompts)}")

# Load checkpoint (for resume)
completed_ids = load_completed_ids(CHECKPOINT_JSONL)
print(f"Already completed: {len(completed_ids)}")

# Filter to remaining prompts
remaining_prompts = [p for p in all_prompts if p['row_id'] not in completed_ids]
print(f"Remaining to process: {len(remaining_prompts)}")

# Show sample
if remaining_prompts:
    print(f"\nSample prompt (first 200 chars):")
    print(remaining_prompts[0]['prompt'][:200] + "...")

## Cell 4: Run Inference

In [ ]:
def save_response(filepath, row_id, response):
    """Append a single response to the checkpoint file."""
    with open(filepath, 'a') as f:
        entry = {'row_id': row_id, 'judge_response': response}
        f.write(json.dumps(entry) + '\n')


def run_inference(prompts, batch_size=BATCH_SIZE):
    """Run inference on all prompts with batching and checkpointing."""
    
    total_batches = (len(prompts) + batch_size - 1) // batch_size
    processed = 0
    errors = 0
    
    print(f"\nProcessing {len(prompts)} prompts in {total_batches} batches (batch_size={batch_size})")
    print("="*60)
    
    for batch_idx in tqdm(range(total_batches), desc="Batches"):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(prompts))
        batch = prompts[start_idx:end_idx]
        
        # Extract prompts for this batch
        batch_prompts = [p['prompt'] for p in batch]
        batch_ids = [p['row_id'] for p in batch]
        
        # Run inference
        try:
            responses = batch_inference(batch_prompts)
            
            # Save each response
            for row_id, response in zip(batch_ids, responses):
                if isinstance(response, Exception):
                    logger.error(f"Failed: {row_id} - {response}")
                    save_response(CHECKPOINT_JSONL, row_id, f"ERROR: {str(response)}")
                    errors += 1
                else:
                    save_response(CHECKPOINT_JSONL, row_id, response)
                    processed += 1
                    
        except Exception as e:
            logger.error(f"Batch {batch_idx} failed entirely: {e}")
            for row_id in batch_ids:
                save_response(CHECKPOINT_JSONL, row_id, f"ERROR: {str(e)}")
                errors += 1
        
        # Progress update every 10 batches
        if (batch_idx + 1) % 10 == 0:
            logger.info(f"Progress: {processed + errors}/{len(prompts)} ({processed} success, {errors} errors)")
    
    print("\n" + "="*60)
    print(f"COMPLETE: {processed} successful, {errors} errors")
    return processed, errors


# Run inference on remaining prompts
if remaining_prompts:
    processed, errors = run_inference(remaining_prompts)
else:
    print("No prompts to process (all already completed)")

## Cell 5: Save Final Output

In [ ]:
import shutil

# Copy checkpoint to final output
if Path(CHECKPOINT_JSONL).exists():
    shutil.copy(CHECKPOINT_JSONL, OUTPUT_JSONL)
    
    # Count results
    results = load_jsonl(OUTPUT_JSONL)
    success_count = sum(1 for r in results if not r['judge_response'].startswith('ERROR:'))
    error_count = len(results) - success_count
    
    print("="*60)
    print("FINAL OUTPUT")
    print("="*60)
    print(f"Output file: {OUTPUT_JSONL}")
    print(f"Total responses: {len(results)}")
    print(f"Successful: {success_count}")
    print(f"Errors: {error_count}")
    print("\n" + "="*60)
    print("NEXT STEP: Download the output file and run:")
    print(f"  python match_error_localization_results.py \\")
    print(f"    --prompts {INPUT_JSONL} \\")
    print(f"    --responses {OUTPUT_JSONL} \\")
    print(f"    --output agreement_results.json")
    print("="*60)
else:
    print("No checkpoint file found - run inference first (Cell 4)")

## Cell 6: Preview Results (Optional)

In [ ]:
# Preview a few responses
if Path(OUTPUT_JSONL).exists():
    results = load_jsonl(OUTPUT_JSONL)
    
    print("Sample responses:")
    print("="*60)
    for i, r in enumerate(results[:3]):
        print(f"\n[{i+1}] row_id: {r['row_id']}")
        response = r['judge_response']
        if len(response) > 300:
            print(f"Response: {response[:300]}...")
        else:
            print(f"Response: {response}")
        print("-"*40)
else:
    print("No output file yet")